In [1]:
from pyspark.sql import SparkSession as ss

spark = ss.builder\
    .appName('calc')\
     .master("local[*]") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio-storage:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()



:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-03b07298-3889-45d8-af34-ae2b86847c4c;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 193ms :: artifacts dl 6ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evic

In [2]:
import pyspark.sql.functions as sf

df = spark.read.parquet("s3a://test-bucket/silver/ecommerce_refined")
df.printSchema()
print("row count:", df.count())
df.groupBy('event_type').count().show()


26/07/31 02:23:38 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- event_date: date (nullable = true)
 |-- category_code: string (nullable = true)



row count: 59442


+----------+-----+
|event_type|count|
+----------+-----+
|  purchase| 1024|
|      view|57493|
|      cart|  925|
+----------+-----+



In [3]:
# 1. 이벤트 퍼널 (view -> cart -> purchase 전환율)
funnel = df.groupBy('event_type').count().collect()
counts = {row['event_type']: row['count'] for row in funnel}

view_cnt = counts.get('view', 0)
cart_cnt = counts.get('cart', 0)
purchase_cnt = counts.get('purchase', 0)

print(f"view     : {view_cnt}")
print(f"cart     : {cart_cnt}" + (f"  (view->cart 전환율: {cart_cnt/view_cnt*100:.2f}%)" if view_cnt else ""))
print(f"purchase : {purchase_cnt}" + (f"  (cart->purchase 전환율: {purchase_cnt/cart_cnt*100:.2f}%)" if cart_cnt else ""))
print(f"전체 전환율 (view->purchase): {purchase_cnt/view_cnt*100:.2f}%" if view_cnt else "")


view     : 57493
cart     : 925  (view->cart 전환율: 1.61%)
purchase : 1024  (cart->purchase 전환율: 110.70%)
전체 전환율 (view->purchase): 1.78%


In [4]:
# 2. 카테고리 / 브랜드별 매출 Top 10 (purchase 이벤트 기준)
purchase_df = df.filter(sf.col('event_type') == 'purchase')

category_revenue = purchase_df.groupBy('category_code') \
    .agg(
        sf.sum('price').alias('revenue'),
        sf.count('*').alias('purchase_count')
    ) \
    .orderBy(sf.desc('revenue'))

print("=== 카테고리별 매출 Top 10 ===")
category_revenue.show(10, truncate=False)

brand_revenue = purchase_df.groupBy('brand') \
    .agg(
        sf.sum('price').alias('revenue'),
        sf.count('*').alias('purchase_count')
    ) \
    .orderBy(sf.desc('revenue'))

print("=== 브랜드별 매출 Top 10 ===")
brand_revenue.show(10, truncate=False)


=== 카테고리별 매출 Top 10 ===


+--------------------------------+------------------+--------------+
|category_code                   |revenue           |purchase_count|
+--------------------------------+------------------+--------------+
|electronics.smartphone          |266180.2100000007 |624           |
|electronics.video.tv            |18423.510000000002|44            |
|appliances.kitchen.washer       |9656.890000000001 |35            |
|computers.notebook              |8275.089999999998 |15            |
|electronics.clocks              |7579.7            |29            |
|electronics.audio.headphone     |7323.759999999997 |78            |
|appliances.kitchen.refrigerators|7107.219999999999 |19            |
|computers.desktop               |4378.68           |13            |
|electronics.tablet              |3149.6000000000004|13            |
|appliances.environment.vacuum   |2109.14           |21            |
+--------------------------------+------------------+--------------+
only showing top 10 rows

=== 브랜드별

+-------+------------------+--------------+
|brand  |revenue           |purchase_count|
+-------+------------------+--------------+
|apple  |182803.81000000008|249           |
|samsung|80817.22999999986 |329           |
|xiaomi |18479.049999999985|111           |
|huawei |8662.999999999996 |45            |
|oppo   |6510.689999999999 |24            |
|acer   |6002.0199999999995|11            |
|lg     |5842.489999999999 |13            |
|indesit|4729.22           |19            |
|lenovo |2384.74           |6             |
|sony   |2323.51           |2             |
+-------+------------------+--------------+
only showing top 10 rows



In [5]:
# 3. 일별(event_date) 추이
daily_events = df.groupBy('event_date', 'event_type') \
    .count() \
    .orderBy('event_date', 'event_type')

print("=== 일별 이벤트 타입별 건수 ===")
daily_events.show(30, truncate=False)

daily_purchase = purchase_df.groupBy('event_date') \
    .agg(
        sf.count('*').alias('purchase_count'),
        sf.sum('price').alias('daily_revenue')
    ) \
    .orderBy('event_date')

print("=== 일별 구매 건수 / 매출 ===")
daily_purchase.show(30, truncate=False)


=== 일별 이벤트 타입별 건수 ===


+----------+----------+-----+
|event_date|event_type|count|
+----------+----------+-----+
|2019-11-01|cart      |925  |
|2019-11-01|purchase  |1024 |
|2019-11-01|view      |57493|
+----------+----------+-----+

=== 일별 구매 건수 / 매출 ===


+----------+--------------+----------------+
|event_date|purchase_count|daily_revenue   |
+----------+--------------+----------------+
|2019-11-01|1024          |348293.190000001|
+----------+--------------+----------------+



In [6]:
# 4. 가격 분포 / 기술통계
df.select('price').describe().show()

df.select(
    sf.expr('percentile_approx(price, 0.25)').alias('p25'),
    sf.expr('percentile_approx(price, 0.5)').alias('median'),
    sf.expr('percentile_approx(price, 0.75)').alias('p75'),
    sf.expr('percentile_approx(price, 0.95)').alias('p95'),
).show()

# purchase 건만 따로 (실제 판매 가격대 분포)
print("=== purchase 이벤트만의 가격 분포 ===")
purchase_df.select('price').describe().show()


+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|            59442|
|   mean|345.9591526193708|
| stddev|375.8301847476457|
|    min|             0.88|
|    max|          2574.07|
+-------+-----------------+



+------+------+------+-------+
|   p25|median|   p75|    p95|
+------+------+------+-------+
|106.54|212.08|446.67|1091.33|
+------+------+------+-------+

=== purchase 이벤트만의 가격 분포 ===


+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|              1024|
|   mean|340.13006835937597|
| stddev|340.86323633037665|
|    min|               7.7|
|    max|           2011.63|
+-------+------------------+



26/07/31 03:32:40 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 530281 ms exceeds timeout 120000 ms
